In [1]:
from pathlib import Path
import sys

day2_root = Path(r"C:\Users\Lokender.Singh\genai-testing-project\interview-prep\day2-ai-testing")
sys.path.append(str(day2_root))

from app.tool_registry import (
    build_default_registry,
    ToolSchemaError,
    ToolDisabledError,
    ToolPermissionError,
    ToolAlreadyExistsError,
    ToolNotFoundError,
)

registry = build_default_registry()

In [2]:
registry.list_tools(role="tester")

[{'name': 'calculator',
  'description': 'Calculate a simple arithmetic expression.',
  'inputSchema': {'type': 'object',
   'properties': {'expression': {'type': 'string',
     'description': "Arithmetic expression, for example '15 * 7'."}},
   'required': ['expression'],
   'additionalProperties': False},
  'enabled': True,
  'allowed_roles': ['admin', 'tester']},
 {'name': 'customer_lookup',
  'description': 'Look up customer details by customer id.',
  'inputSchema': {'type': 'object',
   'properties': {'customer_id': {'type': 'string',
     'description': "Customer id, for example 'CUST-101'."}},
   'required': ['customer_id'],
   'additionalProperties': False},
  'enabled': True,
  'allowed_roles': ['admin', 'tester']},
 {'name': 'ticket_creation',
  'description': 'Create a support ticket.',
  'inputSchema': {'type': 'object',
   'properties': {'title': {'type': 'string',
     'description': 'Short issue title.'},
    'priority': {'type': 'string',
     'enum': ['low', 'medium',

In [3]:
registry.execute_tool(
    "calculator",
    {"expression": "15 * 7"},
    role="tester"
)

{'success': True,
 'tool_name': 'calculator',
 'data': {'expression': '15 * 7', 'result': 105},
 'error': None}

In [4]:
try:
    registry.execute_tool("calculator", {}, role="tester")
except ToolSchemaError as e:
    print("Schema error:", e)

Schema error: 'expression' is a required property

Failed validating 'required' in schema:
    {'type': 'object',
     'properties': {'expression': {'type': 'string',
                                   'description': 'Arithmetic expression, '
                                                  "for example '15 * 7'."}},
     'required': ['expression'],
     'additionalProperties': False}

On instance:
    {}


In [5]:
try:
    registry.execute_tool(
        "calculator",
        {"expression": "15 * 7", "extra": "bad"},
        role="tester"
    )
except ToolSchemaError as e:
    print("Schema error:", e)

Schema error: Additional properties are not allowed ('extra' was unexpected)

Failed validating 'additionalProperties' in schema:
    {'type': 'object',
     'properties': {'expression': {'type': 'string',
                                   'description': 'Arithmetic expression, '
                                                  "for example '15 * 7'."}},
     'required': ['expression'],
     'additionalProperties': False}

On instance:
    {'expression': '15 * 7', 'extra': 'bad'}


In [6]:
registry = build_default_registry()

registry.disable_tool("calculator")

try:
    registry.execute_tool(
        "calculator",
        {"expression": "15 * 7"},
        role="tester"
    )
except ToolDisabledError as e:
    print("Disabled error:", e)

Disabled error: Tool is disabled: calculator


In [7]:
registry.enable_tool("calculator")

registry.execute_tool(
    "calculator",
    {"expression": "15 * 7"},
    role="tester"
)

{'success': True,
 'tool_name': 'calculator',
 'data': {'expression': '15 * 7', 'result': 105},
 'error': None}

In [8]:
try:
    registry.execute_tool(
        "customer_lookup",
        {"customer_id": "CUST-101"},
        role="viewer"
    )
except ToolPermissionError as e:
    print("Permission error:", e)

Permission error: Role 'viewer' cannot execute tool 'customer_lookup'


In [9]:
try:
    registry.execute_tool(
        "delete_all_users",
        {},
        role="admin"
    )
except ToolNotFoundError as e:
    print("Not found error:", e)

Not found error: Tool not registered: delete_all_users


In [10]:
from app.mcp_schemas import CALCULATOR_SCHEMA
from app.tools import calculator_tool

registry = build_default_registry()

try:
    registry.register_tool(CALCULATOR_SCHEMA, calculator_tool)
except ToolAlreadyExistsError as e:
    print("Duplicate error:", e)

Duplicate error: Tool already registered: calculator


In [11]:
registry = build_default_registry()
registry.disable_tool("calculator")

tools = registry.list_tools(role="tester")
[name["name"] for name in tools]

['customer_lookup', 'ticket_creation', 'knowledge_search']

In [12]:
tools = registry.list_tools(role="tester", include_disabled=True)
[(tool["name"], tool["enabled"]) for tool in tools]

[('calculator', False),
 ('customer_lookup', True),
 ('ticket_creation', True),
 ('knowledge_search', True)]